# **Asynchronous Programming for AI Agents**

1. Why AI Agents Need Asynchronous Programming

    I/O-Bound Bottlenecks: Agentic systems spend most of their time waiting on external I/O tasks—such as LLM generation APIs, vector database queries, search engine scraping, and tool calls [05:27].

    Independent Execution in Multi-Agent Grids: When an orchestrator breaks a problem into multiple sub-agents (e.g., Agent 1 fetches weather, Agent 2 fetches news), running them synchronously forces the system to wait for each agent sequentially [07:07].

    Dramatic Latency Reduction: Running independent agent tasks concurrently allows the orchestrator to resolve operations in roughly the time of the single slowest task rather than the sum of all individual runtimes [09:35].

2. Core Concepts: Synchronous vs. Asynchronous

    Subroutines (Synchronous): Standard Python functions execute linearly. When a subroutine triggers an API call and waits for a response, execution halts completely until the response returns [13:09].

    Coroutines (Asynchronous): Declared with async def, coroutines can yield execution control back to Python’s event loop using await whenever an I/O wait occurs, allowing other coroutines to execute in the meantime [15:59].

    Concurrency vs. Parallelism:

        Concurrency: Managing multiple tasks whose execution windows overlap on a single thread/core via rapid context switching while waiting on I/O [32:23].

        Parallelism: Executing multiple CPU-bound tasks at the exact same physical instant across separate CPU cores or processes [31:37].

This notebook introduces **asynchronous programming (`asyncio`)** from the perspective of AI agents.

You will learn:

- Why async matters for AI agents
- `async def`
- `await`
- `asyncio.gather`
- `asyncio.create_task`
- Sequential vs concurrent execution
- Timeouts and error handling
- A simple multi-tool AI-agent pattern

---

## Core idea

AI agents often spend time waiting for:

- LLM APIs
- Web searches
- Databases
- Vector stores
- External tools
- Other agents

If independent operations are executed sequentially:

\[
T = T_1 + T_2 + T_3
\]

If they run concurrently:

\[
T \approx \max(T_1, T_2, T_3)
\]

That can make an agent much faster.

## 1. Synchronous programming

First, consider normal blocking Python code.

Each function must finish before the next one starts.

In [1]:
import time

def get_weather():
    print("Getting weather...")
    time.sleep(2)
    return "Sunny"

def search_database():
    print("Searching database...")
    time.sleep(3)
    return "10 documents found"

def web_search():
    print("Searching web...")
    time.sleep(2)
    return "5 web results"

start = time.time()

weather = get_weather()
documents = search_database()
web_results = web_search()

print("\nResults:")
print(weather)
print(documents)
print(web_results)

print(f"\nTotal time: {time.time() - start:.2f} seconds")

Getting weather...
Searching database...
Searching web...

Results:
Sunny
10 documents found
5 web results

Total time: 7.02 seconds


Expected runtime:

\[
2 + 3 + 2 \approx 7 \text{ seconds}
\]

The three operations are independent, so executing them sequentially is unnecessary.

## 2. First asynchronous example

We replace:

```python
def
```

with:

```python
async def
```

and blocking waits such as:

```python
time.sleep()
```

with:

```python
await asyncio.sleep()
```

In [3]:
import asyncio
import time

async def get_weather_async():
    print("Getting weather...")
    await asyncio.sleep(2)
    return "Sunny"

async def search_database_async():
    print("Searching database...")
    await asyncio.sleep(3)
    return "10 documents found"

async def web_search_async():
    print("Searching web...")
    await asyncio.sleep(2)
    return "5 web results"

start = time.time()

weather, documents, web_results = await asyncio.gather(
    get_weather_async(),
    search_database_async(),
    web_search_async()
)

print("\nResults:")
print(weather)
print(documents)
print(web_results)

print(f"\nTotal time: {time.time() - start:.2f} seconds")

Getting weather...
Searching database...
Searching web...

Results:
Sunny
10 documents found
5 web results

Total time: 3.01 seconds


Expected runtime:

\[
\max(2,3,2) \approx 3 \text{ seconds}
\]

All three operations spend most of their time waiting, so the event loop can switch between them.

## 3. What is a coroutine?

An `async def` function creates a **coroutine function**.

Calling it does not immediately execute the body.

In [4]:
async def hello():
    return "Hello from async Python"

coroutine_object = hello()

print(coroutine_object)

<coroutine object hello at 0x0000028195CB59A0>


To actually run the coroutine inside Jupyter, use:

```python
await hello()
```

In [5]:
result = await hello()
print(result)

Hello from async Python


## 4. Understanding `await`

When Python sees:

```python
result = await some_async_operation()
```

the current coroutine pauses until that operation completes.

While it is waiting, the event loop can execute other asynchronous tasks.

A useful mental model:

```text
Task A starts
    |
    v
await network request
    |
    +------ Task A pauses
               |
               v
          Task B runs
               |
               v
          Task C runs
               |
               v
Network response arrives
    |
    v
Task A resumes
```

## 5. Sequential async is still sequential

Using `async` does not automatically make things concurrent.

This example is asynchronous, but the calls still happen one after another.

In [ ]:
async def operation(name, seconds):
    print(f"{name} started")
    await asyncio.sleep(seconds)
    print(f"{name} finished")
    return name

start = time.time()

a = await operation("A", 2)
b = await operation("B", 3)
c = await operation("C", 1)

print(f"Total time: {time.time() - start:.2f} seconds")

The runtime is approximately:

\[
2 + 3 + 1 = 6 \text{ seconds}
\]

because each `await` waits before starting the next operation.

## 6. Concurrent execution with `asyncio.gather`

`asyncio.gather` is one of the most useful tools for AI-agent workflows.

In [ ]:
start = time.time()

results = await asyncio.gather(
    operation("A", 2),
    operation("B", 3),
    operation("C", 1),
)

print("\nResults:", results)
print(f"Total time: {time.time() - start:.2f} seconds")

Expected runtime:

\[
\max(2,3,1) \approx 3 \text{ seconds}
\]

This is ideal when several tool calls are independent.

## 7. `asyncio.create_task`

`create_task()` schedules a coroutine to run concurrently and gives us a task object.

This is useful when the workflow is more complex than a single `gather()`.

In [ ]:
start = time.time()

task_a = asyncio.create_task(operation("Task A", 2))
task_b = asyncio.create_task(operation("Task B", 3))
task_c = asyncio.create_task(operation("Task C", 1))

print("Tasks have been scheduled.")

result_a = await task_a
result_b = await task_b
result_c = await task_c

print("\nResults:", result_a, result_b, result_c)
print(f"Total time: {time.time() - start:.2f} seconds")

## 8. Agent-style tool example

Imagine an AI agent receives the question:

> "Tell me about working at CERN."

It may want to gather information from several independent tools:

- research search
- job search
- location search

Those can run concurrently.

In [ ]:
async def search_research(topic):
    print("Research tool started...")
    await asyncio.sleep(2)
    return f"Research information about {topic}"

async def search_jobs(topic):
    print("Jobs tool started...")
    await asyncio.sleep(3)
    return f"Job information about {topic}"

async def search_location(topic):
    print("Location tool started...")
    await asyncio.sleep(1)
    return f"Location information about {topic}"

async def agent(question):
    research, jobs, location = await asyncio.gather(
        search_research(question),
        search_jobs(question),
        search_location(question),
    )

    context = f'''
Research:
{research}

Jobs:
{jobs}

Location:
{location}
'''

    return context

start = time.time()

answer_context = await agent("Working at CERN")

print("\nCombined context:")
print(answer_context)

print(f"Total time: {time.time() - start:.2f} seconds")

## 9. Dependency matters

Not every operation should run concurrently.

Consider a RAG pipeline:

```text
retrieve documents
       |
       v
rerank documents
       |
       v
generate answer
```

The reranker needs the retrieved documents, and the generator needs the reranked documents.

Therefore these steps must remain sequential.

In [ ]:
async def retrieve_documents():
    print("Retrieving documents...")
    await asyncio.sleep(1)
    return ["doc1", "doc2", "doc3"]

async def rerank_documents(documents):
    print("Reranking documents...")
    await asyncio.sleep(1)
    return list(reversed(documents))

async def generate_answer(documents):
    print("Generating answer...")
    await asyncio.sleep(1)
    return f"Answer generated from: {documents}"

documents = await retrieve_documents()
ranked_documents = await rerank_documents(documents)
answer = await generate_answer(ranked_documents)

print(answer)

### Rule of thumb

Ask:

> Does operation B require the output of operation A?

If yes:

```text
A -> B
```

Use sequential `await`.

If no:

```text
A --\
B ----> combine results
C --/
```

Concurrent execution may be appropriate.

## 10. Error handling with `asyncio.gather`

Real AI-agent tools fail.

One failing API should not necessarily destroy the entire workflow.

In [ ]:
async def successful_tool():
    await asyncio.sleep(1)
    return "Tool succeeded"

async def failing_tool():
    await asyncio.sleep(1)
    raise RuntimeError("Simulated API failure")

results = await asyncio.gather(
    successful_tool(),
    failing_tool(),
    return_exceptions=True
)

for result in results:
    if isinstance(result, Exception):
        print("Tool failed:", result)
    else:
        print("Tool result:", result)

Without `return_exceptions=True`, the exception normally propagates from `gather()`.

In production agents, you commonly need:

- retries
- fallbacks
- logging
- exception handling
- rate-limit handling

## 11. Timeouts

An agent should not wait forever for a slow API.

In [ ]:
async def very_slow_tool():
    print("Slow tool started...")
    await asyncio.sleep(10)
    return "Finished"

try:
    result = await asyncio.wait_for(
        very_slow_tool(),
        timeout=2
    )
    print(result)

except asyncio.TimeoutError:
    print("Tool timed out.")

Timeouts are especially important for production agents that depend on unreliable external services.

## 12. Simulated multi-agent system

Suppose three specialized agents independently analyze a question:

- Physics Agent
- Machine Learning Agent
- Quantum Agent

They can run concurrently.

In [ ]:
async def physics_agent(question):
    await asyncio.sleep(2)
    return f"Physics analysis of: {question}"

async def ml_agent(question):
    await asyncio.sleep(3)
    return f"Machine-learning analysis of: {question}"

async def quantum_agent(question):
    await asyncio.sleep(2)
    return f"Quantum-computing analysis of: {question}"

question = "Quantum machine learning for particle physics"

start = time.time()

physics, ml, quantum = await asyncio.gather(
    physics_agent(question),
    ml_agent(question),
    quantum_agent(question),
)

print(physics)
print(ml)
print(quantum)

print(f"\nTotal time: {time.time() - start:.2f} seconds")

This pattern appears frequently in:

- supervisor-worker agents
- research agents
- planning agents
- multi-agent frameworks
- LangGraph graphs
- tool-using LLM agents

## 13. Async is not CPU parallelism

`asyncio` is mainly useful for **I/O-bound workloads**.

Good examples:

- LLM API calls
- HTTP requests
- database queries
- web searches
- vector database retrieval
- remote tools

Less suitable examples:

- neural-network training
- large matrix multiplication
- heavy NumPy/Python computation
- local quantum simulation
- CPU-heavy data processing

For CPU-heavy workloads, you may need:

- multiprocessing
- GPUs
- distributed computing
- Ray
- Dask

## 14. Common mistake: blocking inside an async function

This is bad:

```python
async def bad_function():
    time.sleep(5)
```

`time.sleep()` blocks the whole event-loop thread.

Use:

```python
async def good_function():
    await asyncio.sleep(5)
```

In [ ]:
async def good_function():
    print("Started")
    await asyncio.sleep(1)
    print("Finished")

await good_function()

## 15. Connection to LangChain and LangGraph

AI frameworks often provide synchronous and asynchronous APIs.

Typical naming patterns are:

```python
llm.invoke(...)
await llm.ainvoke(...)

chain.batch(...)
await chain.abatch(...)

chain.stream(...)
chain.astream(...)
```

The leading `a` usually means **asynchronous**.

A simplified pattern is:

```python
async def ask_llm(llm, prompt):
    response = await llm.ainvoke(prompt)
    return response
```

Several independent calls can then be executed concurrently:

```python
results = await asyncio.gather(
    ask_llm(llm, "Analyze physics"),
    ask_llm(llm, "Analyze machine learning"),
    ask_llm(llm, "Analyze quantum computing")
)
```

## 16. Final mental model

Remember these four patterns.

### Define a coroutine

```python
async def function():
    ...
```

### Wait for asynchronous work

```python
result = await function()
```

### Run independent operations concurrently

```python
results = await asyncio.gather(
    function1(),
    function2(),
    function3()
)
```

### Schedule a task explicitly

```python
task = asyncio.create_task(function())
result = await task
```

---

## Main principle

\[
\boxed{
\text{Async Agent}
=
\text{Do useful work while other operations are waiting}
}
\]

Before parallelizing agent steps, always ask:

> Are these operations independent?

If they depend on each other's outputs, keep them sequential.

# Exercises

Try these before moving on.

### Exercise 1

Create three asynchronous functions:

- `search_arxiv()` → waits 2 seconds
- `search_vector_db()` → waits 1 second
- `search_web()` → waits 3 seconds

Run them concurrently and measure the total runtime.

### Exercise 2

Modify one tool so it raises an exception.

Use:

```python
return_exceptions=True
```

so the remaining tools can still return results.

### Exercise 3

Create a timeout of 2 seconds for a function that sleeps for 5 seconds.

### Exercise 4

Build this mini agent:

```text
                User Question
                      |
                      v
             +----------------+
             | Research Agent |
             +----------------+
                /     |     \
               /      |      \
          arXiv    VectorDB   Web
               \      |      /
                \     |     /
                 Combine
                    |
                    v
                  Answer
```

The three retrieval tools should run concurrently.